In [0]:
# --------------------------------------------------------------
# Set environment variables for the pipeline run
# --------------------------------------------------------------
# These values are retrieved from the master pipeline task 
# (superstore_pipeline_master_run_id_init) using dbutils.jobs.taskValues.
# They ensure that the ETL, Mart, Features, KPI, and Dashboard code
# runs in the correct environment (dev/qa/prod) without hardcoding paths.
# SUPERSTORE_ENV, SUPERSTORE_PIPELINE_NAME, and SUPERSTORE_PIPELINE_VERSION
# are set for downstream modules to dynamically resolve catalog/schema.
# --------------------------------------------------------------
import os

env = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_ENV")
pipeline_name = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_NAME")
pipeline_version = dbutils.jobs.taskValues.get(taskKey="superstore_pipeline_master_run_id_init", key="SUPERSTORE_PIPELINE_VERSION")

os.environ["SUPERSTORE_ENV"] = env
os.environ["SUPERSTORE_PIPELINE_NAME"] = pipeline_name
os.environ["SUPERSTORE_PIPELINE_VERSION"] = pipeline_version

# -------------------------------
# Environment Config
# -------------------------------
import sys
sys.path.append("/Workspace/Users/bireshmoktan@gmail.com/superstore_medallionarchitecture_dab/src/superstore_shared_utilities")
from superstore_platform_config import get_catalog, get_gold_schema, get_mart_schema, get_features_schema

# Dynamically get environment-aware catalog and schemas
catalog = get_catalog()            # e.g., superstore_catalog
gold_schema = get_gold_schema()    # e.g., dev_gold
features_schema = get_features_schema() 

# -------------------------------
# Create feature customer_product Table (Environment-aware)
# -------------------------------
spark.sql(f"""
-- Create Customer ML Features Table
CREATE OR REPLACE TABLE {catalog}.{features_schema}.features_customer_product AS

SELECT
    s.customer_id,
    s.product_id,

    COUNT(DISTINCT s.order_id) AS purchase_count,

    SUM(s.sales) AS total_spent,
    SUM(s.profit) AS total_profit,

    MAX(o.order_date) AS last_purchase_date,
    DATEDIFF(current_date(), MAX(o.order_date)) AS days_since_last_purchase,

    AVG(s.sales) AS avg_sales_per_order,

    COUNT(DISTINCT CASE 
        WHEN o.order_date >= DATE_SUB(current_date(), 30) 
        THEN s.order_id 
    END) AS orders_last_30d

FROM {catalog}.{gold_schema}.facts_sales s
JOIN {catalog}.{gold_schema}.facts_orders o
    ON s.order_id = o.order_id

GROUP BY 
    s.customer_id, 
    s.product_id;
""")
